# Week 5 Coding Practice: Predicting Outcomes and Finding Hidden Structure

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/DataScience-book/blob/main/module05/week5_trees_factor_analysis_practice.ipynb)

**Estimated time:** 45–60 minutes  
**Format:** worked examples, reasoning checkpoints, and one optional transfer task

This activity contrasts two ways to work with multivariate data. In Part A, we use a known outcome to predict disease progression with decision trees and a random forest. In Part B, we use measurements alone to look for possible latent structure in wine chemistry. These are instructional analyses, not clinical or product decisions.

## Learning objectives

By the end of this activity, you should be able to:

1. translate a prediction question into a target, predictors, and held-out evaluation plan;
2. fit, visualize, and interpret a small regression tree;
3. use train/test evidence to recognize possible overfitting as tree complexity grows;
4. compare one interpretable tree with a random forest on unseen data;
5. interpret feature importance as model evidence, not causal evidence;
6. inspect relationships among observed variables before factor analysis;
7. interpret factor loadings cautiously and propose tentative factor meanings; and
8. distinguish supervised prediction from unsupervised latent-structure discovery.

# Part A. Trees for prediction

## 1. Define the prediction question

The bundled diabetes dataset contains baseline measurements for 442 people and a quantitative measure of disease progression one year later. Our instructional question is:

> Given these baseline measurements, how accurately can a model predict the one-year progression measure for a new person?

The progression measure is the **target**. The baseline measurements are **predictors**. A useful model may support prediction, but it does not establish that a predictor causes progression.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.datasets import load_diabetes, load_wine
from sklearn.decomposition import FactorAnalysis
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

diabetes = load_diabetes(as_frame=True)
tree_data = diabetes.data.copy()
tree_data["progression"] = diabetes.target

print(f"Rows: {len(tree_data)} | Predictors: {tree_data.shape[1] - 1}")
display(tree_data.head())
display(tree_data.describe().T[["mean", "std", "min", "max"]].round(2))

### Feature glossary

The predictors are standardized baseline measurements. `bmi` and `bp` are body-mass index and average blood pressure. `s1` through `s6` are six blood-serum measurements. Standardization makes the numbers less intuitive, but it does not change the prediction question.

### Reasoning checkpoint: choose the evidence before fitting

Before continuing, answer briefly:

- What is the target, and when would it be observed?
- Which baseline measurement do you expect a tree might split on first? Why?
- Why must the test rows be held aside before judging whether the model will generalize?

## 2. Split raw rows into training and test data

We reserve 25% of the rows for testing. Every model below sees the same training rows and is evaluated on the same unseen test rows. This makes the comparison fair.

In [ ]:
X = tree_data.drop(columns="progression")
y = tree_data["progression"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows:     {len(X_test)}")
print(f"Training target mean: {y_train.mean():.1f}")
print(f"Test target mean:     {y_test.mean():.1f}")

## 3. Fit and read a small regression tree

A regression tree divides the predictor space with yes/no rules. Each leaf predicts the average training outcome for the people who reached that leaf. We start with a deliberately small tree: depth 3 and at least 12 training rows per leaf. Those settings make the tree readable and reduce the chance that a leaf represents only a few unusual people.

In [ ]:
small_tree = DecisionTreeRegressor(
    max_depth=3, min_samples_leaf=12, random_state=RANDOM_STATE
)
small_tree.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(17, 8))
plot_tree(
    small_tree,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    precision=1,
    fontsize=9,
    ax=ax,
)
ax.set_title("Small regression tree: predicted one-year progression")
plt.show()

print(f"Actual fitted depth: {small_tree.get_depth()}")
print(f"Number of leaves: {small_tree.get_n_leaves()}")

### Quick Check: translate a path

Choose one path from the root to a leaf in the plotted tree. Translate it into plain language:

1. What conditions define the group that reaches the leaf?
2. What progression value does that leaf predict?
3. Why should we avoid treating that value as an exact prediction for every future person in the group?

A tree can make a rule easy to read, but readability alone does not establish that it is accurate on new data.

## 4. Compare a small tree, a deep tree, and a random forest

Next we compare three models on the same split. The deep tree is intentionally allowed to keep splitting until individual training rows can be isolated. The random forest averages many moderately constrained trees. We use MAE and RMSE; lower values are better. RMSE penalizes especially large errors more heavily.

In [ ]:
deep_tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
forest = RandomForestRegressor(
    n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1
)

models = {
    "Small tree (depth 3)": small_tree,
    "Deep tree": deep_tree,
    "Random forest": forest,
}

def regression_scores(model, X_values, y_values):
    predictions = model.predict(X_values)
    return {
        "MAE": mean_absolute_error(y_values, predictions),
        "RMSE": mean_squared_error(y_values, predictions) ** 0.5,
    }

score_rows = []
for name, model in models.items():
    if name != "Small tree (depth 3)":
        model.fit(X_train, y_train)
    train_scores = regression_scores(model, X_train, y_train)
    test_scores = regression_scores(model, X_test, y_test)
    score_rows.append({
        "model": name,
        "train_MAE": train_scores["MAE"],
        "test_MAE": test_scores["MAE"],
        "train_RMSE": train_scores["RMSE"],
        "test_RMSE": test_scores["RMSE"],
    })

score_table = pd.DataFrame(score_rows).set_index("model").round(1)
display(score_table)

fig, ax = plt.subplots(figsize=(8, 4.5))
score_table[["train_RMSE", "test_RMSE"]].plot.bar(ax=ax, color=["#56B4E9", "#D55E00"])
ax.set_ylabel("RMSE (lower is better)")
ax.set_xlabel("Model")
ax.set_title("Training versus unseen-test error")
ax.tick_params(axis="x", rotation=0)
ax.legend(["Training", "Test"], title="Rows used for scoring")
plt.tight_layout()
plt.show()

print(f"Deep tree depth: {deep_tree.get_depth()} | leaves: {deep_tree.get_n_leaves()}")

### Quick Check: diagnose complexity with evidence

Use the table and chart, not just the model names.

- Which model has the lowest *training* RMSE? Is that enough to choose it?
- Which model has the lowest *test* RMSE on this split?
- Does the deep tree show a train/test gap consistent with overfitting? Explain what you see.
- If the random forest performs better on test rows but is harder to explain, what audience or decision stakes might favor the small tree instead?

An AI assistant that says “choose the deepest tree because its training error is lowest” has ignored the held-out evidence.

## 5. Read feature importance cautiously

Tree-based feature importance summarizes how much a feature helped reduce error across that fitted model's splits. It can be useful for investigation, but it does **not** prove a feature causes the outcome, does not reveal every useful predictor, and can shift when the data or correlated predictors change.

In [ ]:
importance_table = pd.DataFrame(
    {
        "small_tree": small_tree.feature_importances_,
        "random_forest": forest.feature_importances_,
    },
    index=X.columns,
).sort_values("random_forest", ascending=True)

display(importance_table.sort_values("random_forest", ascending=False).round(3))

fig, ax = plt.subplots(figsize=(8, 5))
importance_table.plot.barh(ax=ax, color=["#0072B2", "#009E73"])
ax.set_xlabel("Model feature importance")
ax.set_ylabel("Baseline measurement")
ax.set_title("Importance can differ between one tree and many trees")
ax.legend(["Small tree", "Random forest"])
plt.tight_layout()
plt.show()

### Quick Check: critique an importance claim

Suppose an AI summary says: *“The most important feature causes disease progression, so changing it will improve outcomes.”*

Write a correction that distinguishes what the importance values support from what they do not support. Then name one follow-up analysis or study you would need before making a causal claim.

## 6. Optional transfer: change one meaningful tree decision

Choose **one** change: use `max_depth=2` or `max_depth=5`, change `min_samples_leaf`, or change the random seed used for the train/test split. Predict what will happen to training and test error before you run anything. Then fit one new tree and compare it with the small tree using the same metrics.

This is intentionally an open response. It does not feed any later provided cell.

In [ ]:
# Write your one-decision tree comparison here.
# Tip: create a DecisionTreeRegressor, fit it on X_train and y_train,
# then call regression_scores on X_train/y_train and X_test/y_test.

# Part B. Finding possible latent structure

## 7. Ask a different kind of question

Factor analysis does not start with a target to predict. Instead, it asks whether several observed variables may reflect a smaller number of shared, unobserved patterns.

We use the bundled wine-chemistry dataset. It contains laboratory measurements from wines from three cultivars. We will set the cultivar label aside: our question is not “which cultivar will this be?” but “which measurements tend to move together, and what tentative common concepts might they represent?”

We intentionally use a different dataset here. Forcing the prediction dataset into factor analysis would make the latent-structure question less natural.

In [ ]:
wine = load_wine(as_frame=True)
wine_measurements = wine.data.copy()

print(f"Wine observations: {len(wine_measurements)} | Measurements: {wine_measurements.shape[1]}")
display(wine_measurements.head())

correlations = wine_measurements.corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlations, cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Relationships among observed wine measurements")
plt.tight_layout()
plt.show()

### Quick Check: look before fitting

Identify two or three measurements that appear strongly related in the heatmap. Would it be reasonable to investigate whether they reflect a shared pattern? Why?

Correlation alone does not prove a single hidden factor exists. It does provide a reason to inspect whether a smaller set of shared patterns may be useful.

## 8. Fit a simple two-factor model and inspect loadings

The measurements use different units, so we standardize them before fitting. We ask for two factors as a compact instructional example—not as a claim that exactly two factors are the uniquely correct scientific answer. A **loading** describes the relationship between an observed variable and a fitted factor. Large absolute loadings indicate a stronger relationship; the plus/minus direction is arbitrary for a factor and can be reversed without changing the model.

In [ ]:
wine_scaled = StandardScaler().fit_transform(wine_measurements)
factor_model = FactorAnalysis(n_components=2, random_state=RANDOM_STATE)
factor_model.fit(wine_scaled)

loadings = pd.DataFrame(
    factor_model.components_.T,
    index=wine_measurements.columns,
    columns=["Factor 1", "Factor 2"],
)

display(loadings.reindex(loadings.abs().max(axis=1).sort_values(ascending=False).index).round(2))

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(loadings, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Two-factor loading pattern (standardized measurements)")
plt.tight_layout()
plt.show()

### Quick Check: interpret, then qualify

For each factor:

1. Identify the variables with the largest absolute loadings.
2. Propose a *tentative* phrase that might describe what those variables have in common.
3. Explain why that phrase is an interpretation you supply, not a label automatically discovered by the algorithm.

A good answer refers to the observed variables and avoids presenting a factor name as a verified fact.

# Final comparison: Which question are you answering?

Decision trees and random forests are **supervised** here: they learn from a known target, `progression`, and are judged by how well they predict held-out target values. Factor analysis is **unsupervised** here: it uses observed wine measurements without a target and looks for a possible shared structure.

## Final check

> A dataset contains ten measurements about customers.
>
> If your goal is to predict whether a customer will renew a subscription, which method explored in this notebook would be appropriate?
>
> If your goal is instead to investigate whether the ten measurements reflect a smaller number of underlying customer characteristics, which method would be more appropriate?
>
> Explain why.

Before accepting your own or an AI-generated analysis, check:

- [ ] Does the target (if any) match the practical question?
- [ ] Were prediction claims evaluated on held-out data?
- [ ] Does a complex tree improve test performance, rather than merely training performance?
- [ ] Are feature importance claims limited to what the fitted model supports?
- [ ] Are factor labels presented as tentative interpretations grounded in loadings?
- [ ] Does the conclusion distinguish prediction from explanation or causal proof?